In [ ]:
# --- JASON 101 VERSJON 92.2 ---
# NYTT I 92.2:
# - Beholder FULL original struktur fra 92.0
# - INGEN kolonner flyttet
# - INGEN kolonner fjernet
# - INGEN endringer i QUERY-kompatibilitet
# - Støtte for offseason (unngår StopIteration)
# - Nye dobbel-symboler i GIΔ og xGIΔ
# - Season_Data ark
# - Team_Data ark
#
# VIKTIG:
# - Alt eksisterende er beholdt urørt
# - Nye ting er kun lagt til additivt
# - Kolonnerekkefølge er identisk med 92.0

import requests
import pandas as pd
import numpy as np
from google.colab import auth
import gspread
from google.auth import default
import urllib3
import math
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open("Jason 101")

def clean_float(val):
    try:
        return float(val)
    except:
        return 0.0

def get_fdr_multiplier_v87(fdr_score):

    f = round(fdr_score)

    return (
        1.28 if f <= 1 else
        1.20 if f == 2 else
        1.00 if f == 3 else
        0.88 if f == 4 else
        0.82
    )

def calculate_progressive_xp_v87(ps_val, avg_min):

    if avg_min < 30 or ps_val <= 0:
        return 0.0

    return round(max(0.0, (ps_val * 0.10)), 1)

def trend_symbol(diff, threshold):

    if diff > threshold:
        return "📈"

    elif diff < -threshold:
        return "📉"

    else:
        return "➖"

# --- NYTT: dobbeltsymboler for GIΔ og xGIΔ ---

def level_symbol(val):

    if val >= 0.80:
        return "🟢"

    elif val >= 0.45:
        return "🟡"

    else:
        return "🔴"

def combined_trend_symbol(last6, season, threshold):

    level = level_symbol(season)

    diff = last6 - season

    trend = trend_symbol(diff, threshold)

    return f"{level}{trend}"

def apply_full_formatting(ws, df):

    columns = df.columns.tolist()
    num_rows = len(df)

    requests_list = []

    blue_cols = [
        'BxGI/90',
        'xGI/90 (6)',
        'GI/90 (6)',
        'xGI/90 (S)',
        'GI/90 (S)',
        'xGI',
        'xGI (S6)',
        'GI (S6)',
        'BPS/90',
        'Pot (6)',
        'ICT/90'
    ]

    for c_idx, col_name in enumerate(columns):

        if col_name in blue_cols:

            requests_list.append({
                "addConditionalFormatRule": {
                    "rule": {
                        "ranges": [{
                            "sheetId": ws.id,
                            "startRowIndex": 1,
                            "endRowIndex": 1 + num_rows,
                            "startColumnIndex": c_idx,
                            "endColumnIndex": c_idx + 1
                        }],
                        "gradientRule": {
                            "minpoint": {
                                "color": {
                                    "red": 1.0,
                                    "green": 1.0,
                                    "blue": 1.0
                                },
                                "type": "MIN"
                            },
                            "maxpoint": {
                                "color": {
                                    "red": 0.2,
                                    "green": 0.5,
                                    "blue": 0.9
                                },
                                "type": "MAX"
                            }
                        }
                    },
                    "index": 0
                }
            })

    if requests_list:
        sh.batch_update({"requests": requests_list})

def update_worksheet(sheet_name, dataframe):

    print(f"Oppdaterer {sheet_name}...")

    df_clean = dataframe.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
        sh.del_worksheet(ws)
    except:
        pass

    time.sleep(1)

    ws = sh.add_worksheet(
        title=sheet_name,
        rows="2000",
        cols="120"
    )

    ws.append_row(df_clean.columns.tolist())
    ws.append_rows(df_clean.values.tolist())

    ws.freeze(rows=1, cols=3)

    apply_full_formatting(ws, df_clean)

    time.sleep(10)

def run_jason_101_v92_2():

    base_url = "https://fantasy.premierleague.com/api/"

    r = requests.get(
        f"{base_url}bootstrap-static/",
        verify=False
    ).json()

    elements = pd.DataFrame(r['elements'])

    # --- OFFSEASON FIX ---

    try:

        next_gw = next(
            event['id']
            for event in r['events']
            if event['is_next']
        )

    except StopIteration:

        next_gw = 1

    gw_cols = [
        f"PS{next_gw}",
        f"PS{next_gw+1}",
        f"PS{next_gw+2}",
        f"PS{next_gw+3}",
        f"PS{next_gw+4}"
    ]

    xp_cols = [
        f"xP{next_gw}",
        f"xP{next_gw+1}",
        f"xP{next_gw+2}",
        f"xP{next_gw+3}",
        f"xP{next_gw+4}"
    ]

    target_gws = list(range(next_gw, next_gw + 5))

    teams_det = {
        t['id']: {
            'short': t['short_name'],
            'cs_factor': (
                t['strength_defence_home'] +
                t['strength_defence_away']
            ) / 6000
        }
        for t in r['teams']
    }

    fixtures_r = requests.get(
        f"{base_url}fixtures/",
        verify=False
    ).json()

    last_6_gws = list(range(max(1, next_gw - 6), next_gw))

    main_list = []

    season_archive = []

    team_archive = []

    # --- TEAM DATA ---

    for t in r['teams']:

        team_archive.append({

            'Team': t['name'],
            'Short': t['short_name'],

            'Strength Overall Home': t['strength_overall_home'],
            'Strength Overall Away': t['strength_overall_away'],

            'Strength Attack Home': t['strength_attack_home'],
            'Strength Attack Away': t['strength_attack_away'],

            'Strength Defence Home': t['strength_defence_home'],
            'Strength Defence Away': t['strength_defence_away']
        })

    for _, p in elements.iterrows():

        p_id = p['id']
        team_id = p['team']
        pos_id = p['element_type']

        is_injured = p['status'] in ['d', 'i', 's', 'u']

        try:

            summary = requests.get(
                f"{base_url}element-summary/{p_id}/",
                verify=False
            ).json()

            hist_full = pd.DataFrame(
                summary.get('history', [])
            )

        except:
            continue

        if hist_full.empty:
            continue

        hist_s6 = hist_full[
            hist_full['round'].isin(last_6_gws)
        ]

        m6 = hist_s6['minutes'].sum()

        x6 = round(
            hist_s6['expected_goal_involvements']
            .apply(clean_float)
            .sum(),
            2
        )

        g6 = int(
            hist_s6['goals_scored'].sum() +
            hist_s6['assists'].sum()
        )

        ict6 = round(
            hist_s6['ict_index']
            .apply(clean_float)
            .sum(),
            2
        )

        hist_agg = hist_s6.groupby('round').agg({
            'total_points': 'sum',
            'minutes': 'sum',
            'element': 'count'
        }).to_dict('index')

        f_guide = ""
        min_g = ""
        t_sum = 0

        for gw in last_6_gws:

            if any(
                f['event'] == gw and
                (
                    f['team_h'] == team_id or
                    f['team_a'] == team_id
                )
                for f in fixtures_r
            ):

                d = hist_agg.get(gw)

                if d:

                    t_sum += math.sqrt(
                        max(0, d['total_points'])
                    )

                    num_matches = d.get('element', 1)

                    green_limit = 6 * num_matches
                    yellow_limit = 4 * num_matches

                    f_guide += (
                        "🟢"
                        if d['total_points'] >= green_limit else
                        "🟡"
                        if d['total_points'] >= yellow_limit else
                        "🔴"
                        if d['minutes'] > 0 else
                        "⚪"
                    )

                    min_g += (
                        "🟢"
                        if d['minutes'] >= (78 * num_matches) else
                        "🟡"
                        if d['minutes'] >= (60 * num_matches) else
                        "🔴"
                        if d['minutes'] > 0 else
                        "⚪"
                    )

                else:

                    f_guide += "⚪"
                    min_g += "⚪"

            else:

                f_guide += "X"
                min_g += "X"

        v_form = round((t_sum / 6) * 4, 1)

        team_f_6 = [
            f for f in fixtures_r
            if f['event'] in last_6_gws and
            (
                f['team_h'] == team_id or
                f['team_a'] == team_id
            )
        ]

        avg_min_6 = (
            min(90.0, m6 / len(team_f_6))
            if len(team_f_6) > 0 else 0
        )

        gi_90_6 = (
            round((g6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        xgi_90_6 = (
            round((x6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        ict_90_6 = (
            round((ict6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        season_minutes = (
            float(p['minutes'])
            if float(p['minutes']) > 0 else 1.0
        )

        season_gi = int(
            p['goals_scored'] +
            p['assists']
        )

        season_xgi = round(
            clean_float(
                p['expected_goal_involvements']
            ),
            2
        )

        season_ict = round(
            clean_float(p['ict_index']),
            2
        )

        season_gi_90 = round(
            (season_gi / season_minutes) * 90,
            2
        )

        season_xgi_90 = round(
            (season_xgi / season_minutes) * 90,
            2
        )

        season_ict_90 = round(
            (season_ict / season_minutes) * 90,
            2
        )

        # --- NY DOBBELTSYMBOL LOGIKK ---

        gi_trend = combined_trend_symbol(
            gi_90_6,
            season_gi_90,
            0.15
        )

        xgi_trend = combined_trend_symbol(
            xgi_90_6,
            season_xgi_90,
            0.15
        )

        ict_trend = trend_symbol(
            ict_90_6 - season_ict_90,
            1.5
        )

        heat_score = round(
            (
                (gi_90_6 - season_gi_90) +
                (xgi_90_6 - season_xgi_90)
            ) / 2,
            2
        )

        if avg_min_6 < 30:

            heat_symbol = "---"

        else:

            if heat_score >= 0.55:
                heat_symbol = "🔥🔥🔥"

            elif heat_score >= 0.30:
                heat_symbol = "🔥🔥"

            elif heat_score >= 0.10:
                heat_symbol = "🔥"

            elif heat_score > -0.10:
                heat_symbol = "➖"

            elif heat_score > -0.30:
                heat_symbol = "💩"

            else:
                heat_symbol = "💩💩"

        xgi_bal = (
            (xgi_90_6 * 0.3) +
            (
                (
                    season_xgi /
                    season_minutes *
                    90
                ) * 0.7
            )
        )

        base_p = (
            (
                float(p['points_per_game']) *
                10.0 *
                0.60
            ) +
            (
                (
                    (xgi_bal * 45.0) +
                    (v_form * 2.8) +
                    (
                        (
                            p['bps'] /
                            season_minutes *
                            90
                        ) / 5
                    )
                ) * 0.40
            )
        )

        if pos_id == 2:

            base_p += (
                teams_det[team_id]['cs_factor'] *
                22.0
            )

        if (
            (
                p['total_points'] >= 25 or
                next_gw < 20
            ) and
            (m6 >= 90 or next_gw == 1)
        ):

            adj_ps = []

            for gw in target_gws:

                gw_f = [
                    f for f in fixtures_r
                    if f['event'] == gw and
                    (
                        f['team_h'] == team_id or
                        f['team_a'] == team_id
                    )
                ]

                adj_ps.append(
                    sum(
                        base_p *
                        get_fdr_multiplier_v87(
                            float(
                                f['team_h_difficulty']
                                if f['team_h'] == team_id
                                else f['team_a_difficulty']
                            )
                        )
                        for f in gw_f
                    )
                )

            status_symbol = "---"

            p_dict = {

                'Navn': f"{'⚠️ ' if is_injured else ''}{p['first_name']} {p['second_name']}",

                'Lag': teams_det[team_id]['short'],

                'Pos': {
                    1: 'GKP',
                    2: 'DEF',
                    3: 'MID',
                    4: 'FWD'
                }[pos_id],

                'Pris': float(p['now_cost']) / 10,

                'Eie %': p['selected_by_percent'],

                'Form': v_form,

                'Siste 6': f_guide,

                'PPK': float(p['points_per_game']),

                'BPS/90': round(
                    (
                        p['bps'] /
                        season_minutes *
                        90
                    ),
                    1
                ),

                'BxGI/90': round(xgi_bal, 2),

                'PS (N5)': round(sum(adj_ps) / 5, 1),

                'PS Tot': round(sum(adj_ps), 1)
            }

            for i, name in enumerate(gw_cols):

                p_dict[name] = round(adj_ps[i], 1)

            txp = []

            for i, ps_v in enumerate(adj_ps):

                v = calculate_progressive_xp_v87(
                    ps_v,
                    avg_min_6
                )

                p_dict[xp_cols[i]] = v

                txp.append(v)

            p_dict.update({

                'SUM5': round(sum(txp), 1),

                'P6': int(
                    hist_s6['total_points'].sum()
                ),

                'GI': season_gi,

                'xGI': season_xgi,

                'xGI (S6)': x6,

                'GI (S6)': g6,

                'Pot (6)': round(
                    x6 - g6,
                    2
                ),

                'GI/90 (6)': gi_90_6,

                'xGI/90 (6)': xgi_90_6,

                'GI/90 (S)': season_gi_90,

                'xGI/90 (S)': season_xgi_90,

                'HeatScore': heat_score,

                'Heat': heat_symbol,

                'Merknad': (
                    p['news']
                    if is_injured else ""
                ),

                'Total min sesong': p['minutes'],

                'Min S/6': round(avg_min_6, 1),

                'Min-guide': min_g,

                'Status': status_symbol,

                'GIΔ': gi_trend,

                'xGIΔ': xgi_trend,

                'ICT/90': season_ict_90,

                'ICTΔ': ict_trend
            })

            main_list.append(p_dict)

        # --- SEASON DATA ARKIV ---

        season_archive.append({

            'Player ID': p_id,

            'Navn': f"{p['first_name']} {p['second_name']}",

            'Lag': teams_det[team_id]['short'],

            'Pos': {
                1: 'GKP',
                2: 'DEF',
                3: 'MID',
                4: 'FWD'
            }[pos_id],

            'Pris': float(p['now_cost']) / 10,

            'Minutes': p['minutes'],

            'Points': p['total_points'],

            'PPG': p['points_per_game'],

            'GI': season_gi,

            'xGI': season_xgi,

            'GI/90': season_gi_90,

            'xGI/90': season_xgi_90,

            'ICT/90': season_ict_90,

            'BPS': p['bps'],

            'Selected %': p['selected_by_percent']
        })

    df_f = pd.DataFrame(main_list)

    if df_f.empty:

        print("Ingen spillere funnet.")
        return

    # --- ORIGINAL ORDER FRA 92.0 BEHOLDT URØRT ---

    order = [

        'Navn',
        'Lag',
        'Pos',
        'Pris',
        'Eie %',
        'Form',
        'Siste 6',
        'PPK',
        'BPS/90',
        'BxGI/90',
        'PS (N5)',
        'PS Tot'

    ] + gw_cols + xp_cols + [

        'SUM5',
        'P6',
        'GI',
        'xGI',
        'xGI (S6)',
        'GI (S6)',
        'Pot (6)',

        'GI/90 (6)',
        'xGI/90 (6)',

        'GI/90 (S)',
        'xGI/90 (S)',

        'HeatScore',
        'Heat',

        'Merknad',
        'Total min sesong',
        'Min S/6',
        'Min-guide',
        'Status',

        'GIΔ',
        'xGIΔ',
        'ICT/90',
        'ICTΔ'
    ]

    update_worksheet(
        "ICT",
        df_f.sort_values(
            'PS Tot',
            ascending=False
        )[order]
    )

    for s, pos in {
        "Keeper": "GKP",
        "Forsvar": "DEF",
        "Midtbane": "MID",
        "Angrep": "FWD"
    }.items():

        update_worksheet(
            s,
            df_f[df_f['Pos'] == pos]
            .sort_values(
                'PS Tot',
                ascending=False
            )[order]
        )

    update_worksheet(
        "xP",
        df_f.sort_values(
            'SUM5',
            ascending=False
        )[order]
    )

    # --- NYE ARK ---

    update_worksheet(
        "Season_Data",
        pd.DataFrame(season_archive)
    )

    update_worksheet(
        "Team_Data",
        pd.DataFrame(team_archive)
    )

    print(
        "Versjon 92.2 ferdig "
        "med offseason-fix, "
        "dobbeltsymboler, "
        "Season_Data og Team_Data."
    )

run_jason_101_v92_2()

Oppdaterer ICT...
Oppdaterer Keeper...
Oppdaterer Forsvar...
Oppdaterer Midtbane...
Oppdaterer Angrep...
Oppdaterer xP...
Oppdaterer Season_Data...
Oppdaterer Team_Data...
Versjon 92.2 ferdig med offseason-fix, dobbeltsymboler, Season_Data og Team_Data.
